In [1]:
import os
import sys
import pandas as pd

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from utility.data_log_functions import DataLogHelper

In [4]:
def compare_multiple_code_generation_logs(res_dir: str, filter: str = None):

    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv") and filter in f)]

    count = 0
    res = []

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]

    results_df = pd.DataFrame(columns=log_file_names, index = log_file_names)
    for file_name in log_file_names:
        results_df.loc[file_name, file_name] = float('nan')

    while count < len(csv_logs):
        log1_file_name = csv_logs.pop()
        for log2_file_name in csv_logs:
            log1_file_path = os.path.join(res_dir, log1_file_name)
            log2_file_path = os.path.join(res_dir, log2_file_name)

            log1 = pd.read_csv(log1_file_path)
            log2 = pd.read_csv(log2_file_path) 

            log1_inconsistencies, log2_inconsistencies = DataLogHelper.compare_code_inconsistency_dataframe_results(log1=log1, log2=log2)
            print(f"log1: {log1_file_name} > {log1_inconsistencies}")
            print(f"log2: {log2_file_name} > {log2_inconsistencies}")

            results_df.loc[log1_file_name.replace('.csv', ''), log2_file_name.replace('.csv', '')] = log1_inconsistencies
            results_df.loc[log2_file_name.replace('.csv', ''), log1_file_name.replace('.csv', '')] = log2_inconsistencies

        count += 1
    
    return results_df

In [5]:
res_dir = proj_dir + "/results/code_inconsistencies/mistral"

res = compare_multiple_code_generation_logs(res_dir=res_dir, filter="zero_shot")
res_df = pd.DataFrame(res)

print(res_df)

log1: mistral-small-2506_zero_shot_for2while_sequential mutation.csv > 18/283
log2: mistral-small-2506_zero_shot_for2while_random mutation.csv > 24/283
log1: mistral-small-2506_zero_shot_for2while_sequential mutation.csv > 80/345
log2: mistral-small-2506_zero_shot_None_None mutation.csv > 19/345
log1: mistral-small-2506_zero_shot_None_None mutation.csv > 19/345
log2: mistral-small-2506_zero_shot_for2while_random mutation.csv > 86/345
                                                   mistral-small-2506_zero_shot_for2while_random mutation  \
mistral-small-2506_zero_shot_for2while_random m...                                                NaN       
mistral-small-2506_zero_shot_None_None mutation                                                19/345       
mistral-small-2506_zero_shot_for2while_sequenti...                                             18/283       

                                                   mistral-small-2506_zero_shot_None_None mutation  \
mistral-small-2506_zero